# RNV1 — Sub-¥10,000 Autonomous Humanoid Extreme RL Training (Google Colab Pro A100)

**Version:** `RNV1` (Robotic Neural Vector 1)  
**Target Hardware:** Seeed Studio XIAO ESP32S3 Sense + 10× MG90S + 2× SG90 + PCA9685 + MPU-6050 + VL53L0X + MAX98357A  
**Physics Engine:** Google DeepMind **MuJoCo MJX (100% Pure-JAX GPU XLA)** — `4,096` parallel humanoid environments on **NVIDIA A100** (Zero Flax/Optax dependency — JAX 0.4〜0.12+ / Python 3.10〜3.13 完全互換)  
**Output:** Drop-in ESP32-S3 C header `rnv1_policy_weights.h` (`36 → 64 → 64 → 12` MLP + CPG Prior, **29.2 KB Flash**, **< 0.85 ms** @ 50 Hz)

### One-Click Quickstart (ボタンを押すだけ!)
1. Colabメニュー **「ランタイム」→「ランタイムのタイプを変更」→ `A100 GPU`** を選択
2. **「ランタイム」→「すべてのセルを実行 (Run All / Ctrl+F9)」** を押すだけ!
3. 約12〜18分でゼロから **1.2億ステップ (120M steps)** の極限強化学習が完了し、シミュレーション動画とESP32-S3書き込み用ファイル `RNV1_Trained_Bundle.zip` (`rnv1_policy_weights.h` 入り) が自動ダウンロードされます。


In [ ]:
#@title 1. GPU (A100) Verification & XLA Performance Tuning { display-mode: "form" }
import os, subprocess, sys, time

TRAINING_VERSION = "RNV1"
NUM_ENVS = 4096               #@param {type:"integer"}
TOTAL_TIMESTEPS = 120_000_000 #@param {type:"integer"}
SEED = 2026                   #@param {type:"integer"}

os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=true"
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.88"

print(f"=== {TRAINING_VERSION} Extreme RL Pipeline (Google Colab Pro A100 / Pure-JAX) ===")
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"[GPU DETECTED] {gpu_info}")
    if "A100" in gpu_info or "H100" in gpu_info:
        pass
    elif "L4" in gpu_info or "V100" in gpu_info or "A10" in gpu_info:
        print("[NOTE] Mid-tier GPU detected: auto-scaling NUM_ENVS to 2048 for VRAM headroom.")
        NUM_ENVS = min(NUM_ENVS, 2048)
    else:
        print("[NOTE] Entry GPU detected: auto-scaling NUM_ENVS to 1024 for VRAM safety.")
        NUM_ENVS = min(NUM_ENVS, 1024)
except Exception as e:
    print(f"[WARN] No NVIDIA GPU detected ({e}); falling back to CPU smoke-test mode (NUM_ENVS=16).")
    NUM_ENVS = 16
    TOTAL_TIMESTEPS = 32_768


In [ ]:
#@title 2. Auto-Install MuJoCo, MJX & MediaPy (Pure-JAX — Zero Flax/Optax Dependency) { display-mode: "form" }
import importlib, subprocess, sys

def ensure_pkgs():
    to_install = []
    for mod, pkg in [("mujoco", "mujoco>=3.2.0"), ("mujoco.mjx", "mujoco-mjx>=3.2.0"), ("mediapy", "mediapy")]:
        try:
            importlib.import_module(mod)
        except Exception:
            to_install.append(pkg)
    if to_install:
        print("Installing packages:", to_install)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    else:
        print("All MuJoCo-MJX & JAX packages already installed.")

ensure_pkgs()

import jax
import jax.numpy as jnp
import mujoco
from mujoco import mjx
import numpy as np
import matplotlib.pyplot as plt

print(f"JAX {jax.__version__} | Devices: {jax.devices()} | MuJoCo {mujoco.__version__}")


In [ ]:
#@title 3. Materialize Exact CAD-Synchronized RNV1 MuJoCo MJCF Model { display-mode: "form" }
from pathlib import Path

RNV1_MJCF_XML = """<mujoco model="rnv1_humanoid">
  <compiler angle="radian" coordinate="local" inertiafromgeom="false"/>
  <option timestep="0.002" gravity="0 0 -9.81" integrator="implicitfast" iterations="25" ls_iterations="10">
    <flag eulerdamp="enable"/>
  </option>

  <visual>
    <global offwidth="1280" offheight="720"/>
    <rgba haze="1 1 1 1"/>
  </visual>

  <default>
    <joint damping="0.019" armature="0.00045" frictionloss="0.003" limited="true"/>
    <geom contype="0" conaffinity="0" condim="3" friction="0.90 0.015 0.004" solref="0.012 1.0" solimp="0.9 0.95 0.001"/>
    <default class="mg90s_leg">
      <position kp="2.8" kv="0.065" forcerange="-0.206 0.206"/>
    </default>
    <default class="mg90s_arm">
      <position kp="2.4" kv="0.055" forcerange="-0.206 0.206"/>
    </default>
    <default class="sg90_head">
      <position kp="1.6" kv="0.035" forcerange="-0.155 0.155"/>
    </default>
  </default>

  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.96 0.96 0.97" rgb2="0.90 0.91 0.93" width="256" height="256"/>
    <material name="floor_mat" texture="grid" texrepeat="12 12" reflectance="0.05"/>
    <material name="shell_white" rgba="0.95 0.96 0.97 1"/>
    <material name="servo_blue" rgba="0.16 0.36 0.68 1"/>
    <material name="visor_dark" rgba="0.09 0.11 0.15 1"/>
    <material name="sole_rubber" rgba="0.18 0.19 0.21 1"/>
  </asset>

  <worldbody>
    <light pos="0.5 -0.8 1.5" dir="-0.3 0.5 -1" diffuse="0.9 0.9 0.9"/>
    <geom name="floor" type="plane" size="20 20 0.05" material="floor_mat" contype="1" conaffinity="1"/>

    <!-- TORSO + PELVIS UNIBODY (Origin at Z_HIP_ROLL = 0.124 m) -->
    <body name="torso" pos="0 0 0.124">
      <freejoint name="root"/>
      <inertial pos="0.001 0 0.022" mass="0.20219" diaginertia="0.0001857 0.0001270 0.0001570"/>
      <geom name="torso_geom" type="box" pos="0 0 0.026" size="0.027 0.040 0.034" material="shell_white" contype="2" conaffinity="1"/>
      <site name="imu_site" pos="0.002 0 0.012" size="0.004" rgba="1 0.2 0.2 1"/>

      <!-- 2-DOF ACTIVE SPATIAL HEAD -->
      <body name="neck_gimbal" pos="0 0 0.064">
        <inertial pos="0 0 0.009" mass="0.01266" diaginertia="0.0000010 0.0000010 0.0000010"/>
        <joint name="head_yaw" type="hinge" axis="0 0 1" range="-1.309 1.309" damping="0.012"/>
        <geom name="neck_geom" type="cylinder" pos="0 0 0.008" size="0.010 0.009" material="servo_blue"/>

        <body name="head" pos="0 0 0.016">
          <inertial pos="0.003 0 0.013" mass="0.02398" diaginertia="0.0000102 0.0000074 0.0000106"/>
          <joint name="head_pitch" type="hinge" axis="0 1 0" range="-0.524 0.611" damping="0.012"/>
          <geom name="head_geom" type="box" pos="0 0 0.013" size="0.022 0.029 0.021" material="shell_white" contype="2" conaffinity="1"/>
          <geom name="visor_geom" type="box" pos="0.0215 0 0.0135" size="0.002 0.021 0.012" material="visor_dark"/>
          <camera name="head_cam" pos="0.023 0 0.015" xyaxes="0 -1 0 0 0 1" fovy="62"/>
          <site name="tof_site" pos="0.023 0 0.005" zaxis="1 0 0" size="0.002" rgba="0 0.8 1 1"/>
        </body>
      </body>

      <!-- LEFT & RIGHT ARMS -->
      <body name="arm_L" pos="0 0.046 0.046">
        <inertial pos="0.004 0 -0.032" mass="0.00741" diaginertia="0.0000033 0.0000034 0.0000010"/>
        <joint name="l_shoulder" type="hinge" axis="0 1 0" range="-1.658 2.618"/>
        <geom name="arm_L_geom" type="capsule" fromto="0 0 0 0.011 0 -0.068" size="0.006" material="shell_white" contype="2" conaffinity="1"/>
      </body>

      <body name="arm_R" pos="0 -0.046 0.046">
        <inertial pos="0.004 0 -0.032" mass="0.00741" diaginertia="0.0000033 0.0000034 0.0000010"/>
        <joint name="r_shoulder" type="hinge" axis="0 1 0" range="-1.658 2.618"/>
        <geom name="arm_R_geom" type="capsule" fromto="0 0 0 0.011 0 -0.068" size="0.006" material="shell_white" contype="2" conaffinity="1"/>
      </body>

      <!-- LEFT LEG (4-DOF: Hip Roll, Hip Pitch, Coaxial Universal Ankle Pitch + Ankle Roll) -->
      <body name="hip_clevis_L" pos="0 0.023 0">
        <inertial pos="0 0.002 -0.010" mass="0.01907" diaginertia="0.0000017 0.0000018 0.0000017"/>
        <joint name="l_hip_roll" type="hinge" axis="1 0 0" range="-0.524 0.611"/>
        <geom name="hip_L_geom" type="box" pos="0 0.002 -0.009" size="0.011 0.010 0.011" material="servo_blue"/>

        <body name="leg_L" pos="0 0.003 -0.018">
          <!-- Ankle Pitch MG90S mounted at bottom of 88mm leg strut -> lowers CoM! -->
          <inertial pos="0 0 -0.056" mass="0.03596" diaginertia="0.0000249 0.0000243 0.0000028"/>
          <joint name="l_hip_pitch" type="hinge" axis="0 1 0" range="-0.960 0.960"/>
          <geom name="leg_L_geom" type="box" pos="0 0 -0.044" size="0.010 0.011 0.044" material="shell_white"/>

          <body name="ankle_L" pos="0 0 -0.088">
            <inertial pos="0 0 0.002" mass="0.01799" diaginertia="0.0000016 0.0000015 0.0000016"/>
            <joint name="l_ankle_pitch" type="hinge" axis="0 1 0" range="-0.785 0.785"/>
            <geom name="ankle_L_geom" type="box" pos="0 0 0.002" size="0.010 0.010 0.010" material="servo_blue"/>

            <body name="foot_L" pos="0 0 0">
              <inertial pos="0.003 0 -0.013" mass="0.01568" diaginertia="0.0000031 0.0000071 0.0000095"/>
              <joint name="l_ankle_roll" type="hinge" axis="1 0 0" range="-0.611 0.611"/>
              <geom name="foot_L_sole" type="box" pos="0.003 0 -0.015" size="0.024 0.023 0.003" material="sole_rubber" contype="4" conaffinity="1"/>
              <geom name="foot_L_toe" type="box" pos="0.032 0 -0.0135" euler="0 -0.14 0" size="0.008 0.021 0.003" material="shell_white" contype="4" conaffinity="1"/>
              <geom name="foot_L_heel" type="box" pos="-0.025 0 -0.0135" euler="0 0.14 0" size="0.007 0.021 0.003" material="shell_white" contype="4" conaffinity="1"/>
              <site name="l_foot_site" pos="0.003 0 -0.018" size="0.024 0.023 0.002" type="box" rgba="0 1 0 0.0"/>
            </body>
          </body>
        </body>
      </body>

      <!-- RIGHT LEG (4-DOF: Hip Roll, Hip Pitch, Coaxial Universal Ankle Pitch + Ankle Roll) -->
      <body name="hip_clevis_R" pos="0 -0.023 0">
        <inertial pos="0 -0.002 -0.010" mass="0.01907" diaginertia="0.0000017 0.0000018 0.0000017"/>
        <joint name="r_hip_roll" type="hinge" axis="1 0 0" range="-0.611 0.524"/>
        <geom name="hip_R_geom" type="box" pos="0 -0.002 -0.009" size="0.011 0.010 0.011" material="servo_blue"/>

        <body name="leg_R" pos="0 -0.003 -0.018">
          <inertial pos="0 0 -0.056" mass="0.03596" diaginertia="0.0000249 0.0000243 0.0000028"/>
          <joint name="r_hip_pitch" type="hinge" axis="0 1 0" range="-0.960 0.960"/>
          <geom name="leg_R_geom" type="box" pos="0 0 -0.044" size="0.010 0.011 0.044" material="shell_white"/>

          <body name="ankle_R" pos="0 0 -0.088">
            <inertial pos="0 0 0.002" mass="0.01799" diaginertia="0.0000016 0.0000015 0.0000016"/>
            <joint name="r_ankle_pitch" type="hinge" axis="0 1 0" range="-0.785 0.785"/>
            <geom name="ankle_R_geom" type="box" pos="0 0 0.002" size="0.010 0.010 0.010" material="servo_blue"/>

            <body name="foot_R" pos="0 0 0">
              <inertial pos="0.003 0 -0.013" mass="0.01568" diaginertia="0.0000031 0.0000071 0.0000095"/>
              <joint name="r_ankle_roll" type="hinge" axis="1 0 0" range="-0.611 0.611"/>
              <geom name="foot_R_sole" type="box" pos="0.003 0 -0.015" size="0.024 0.023 0.003" material="sole_rubber" contype="4" conaffinity="1"/>
              <geom name="foot_R_toe" type="box" pos="0.032 0 -0.0135" euler="0 -0.14 0" size="0.008 0.021 0.003" material="shell_white" contype="4" conaffinity="1"/>
              <geom name="foot_R_heel" type="box" pos="-0.025 0 -0.0135" euler="0 0.14 0" size="0.007 0.021 0.003" material="shell_white" contype="4" conaffinity="1"/>
              <site name="r_foot_site" pos="0.003 0 -0.018" size="0.024 0.023 0.002" type="box" rgba="0 1 0 0.0"/>
            </body>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <contact>
    <pair geom1="floor" geom2="foot_L_sole" friction="0.90 0.90 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="foot_L_toe"  friction="0.80 0.80 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="foot_L_heel" friction="0.80 0.80 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="foot_R_sole" friction="0.90 0.90 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="foot_R_toe"  friction="0.80 0.80 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="foot_R_heel" friction="0.80 0.80 0.015 0.004 0.004"/>
    <pair geom1="floor" geom2="torso_geom"  friction="0.60 0.60 0.01 0.002 0.002"/>
    <pair geom1="floor" geom2="head_geom"   friction="0.60 0.60 0.01 0.002 0.002"/>
    <pair geom1="floor" geom2="arm_L_geom"  friction="0.80 0.80 0.01 0.002 0.002"/>
    <pair geom1="floor" geom2="arm_R_geom"  friction="0.80 0.80 0.01 0.002 0.002"/>
  </contact>

  <actuator>
    <position name="act_l_hip_roll"    joint="l_hip_roll"    class="mg90s_leg" ctrlrange="-0.524 0.611"/>
    <position name="act_l_hip_pitch"   joint="l_hip_pitch"   class="mg90s_leg" ctrlrange="-0.960 0.960"/>
    <position name="act_l_ankle_pitch" joint="l_ankle_pitch" class="mg90s_leg" ctrlrange="-0.785 0.785"/>
    <position name="act_l_ankle_roll"  joint="l_ankle_roll"  class="mg90s_leg" ctrlrange="-0.611 0.611"/>
    <position name="act_r_hip_roll"    joint="r_hip_roll"    class="mg90s_leg" ctrlrange="-0.611 0.524"/>
    <position name="act_r_hip_pitch"   joint="r_hip_pitch"   class="mg90s_leg" ctrlrange="-0.960 0.960"/>
    <position name="act_r_ankle_pitch" joint="r_ankle_pitch" class="mg90s_leg" ctrlrange="-0.785 0.785"/>
    <position name="act_r_ankle_roll"  joint="r_ankle_roll"  class="mg90s_leg" ctrlrange="-0.611 0.611"/>
    <position name="act_l_shoulder"    joint="l_shoulder"    class="mg90s_arm" ctrlrange="-1.658 2.618"/>
    <position name="act_r_shoulder"    joint="r_shoulder"    class="mg90s_arm" ctrlrange="-1.658 2.618"/>
    <position name="act_head_yaw"      joint="head_yaw"      class="sg90_head" ctrlrange="-1.309 1.309"/>
    <position name="act_head_pitch"    joint="head_pitch"    class="sg90_head" ctrlrange="-0.524 0.611"/>
  </actuator>

  <sensor>
    <gyro name="imu_gyro" site="imu_site"/>
    <accelerometer name="imu_accel" site="imu_site"/>
    <framequat name="imu_quat" objtype="site" objname="imu_site"/>
    <rangefinder name="tof_range" site="tof_site"/>
    <touch name="l_foot_touch" site="l_foot_site"/>
    <touch name="r_foot_touch" site="r_foot_site"/>
  </sensor>

  <keyframe>
    <key name="home" qpos="0 0 0.124 1 0 0 0  0 0  0 0  0 0 0 0  0 0 0 0"/>
  </keyframe>
</mujoco>
"""

Path("rnv1_humanoid.xml").write_text(RNV1_MJCF_XML, encoding="utf-8")
mj_model = mujoco.MjModel.from_xml_path("rnv1_humanoid.xml")
mjx_model = mjx.put_model(mj_model)

print(f"[OK] Loaded RNV1 MJCF | Total mass: {float(np.sum(mj_model.body_mass)):.4f} kg | "
      f"Actuators: {mj_model.nu} | DoF: {mj_model.nv} | MJX GPU Ready!")


In [ ]:
#@title 4. Define Pure-JAX RNV1 (36->64->64->12 + CPG) & 4,096-Env Domain-Randomized MJX Task { display-mode: "form" }
from typing import NamedTuple

OBS_DIM = 36
PRIV_DIM = 52
H1_DIM = 64
H2_DIM = 64
ACT_DIM = 12
CONTROL_HZ = 50.0
CONTROL_DT = 1.0 / CONTROL_HZ
PHYSICS_SUBSTEPS = 10  # 10 x 2ms = 20ms (50Hz)
EPISODE_STEPS = 500    # 10 seconds per episode
RESIDUAL_SCALE = 0.35

Q_MIN = jnp.array([
    -0.524, -0.960, -0.785, -0.611,
    -0.611, -0.960, -0.785, -0.611,
    -1.658, -1.658, -1.309, -0.524,
], dtype=jnp.float32)

Q_MAX = jnp.array([
     0.611,  0.960,  0.785,  0.611,
     0.524,  0.960,  0.785,  0.611,
     2.618,  2.618,  1.309,  0.611,
], dtype=jnp.float32)

DEFAULT_CPG = jnp.array([
    1.45, 0.01, 1.00, 0.19, -0.98, 0.25,
    0.09, 0.00, 0.42, 0.02, 0.65, 0.04
], dtype=jnp.float32)

# Bilateral L/R mirror-reflection involutions for symmetry-regularized PPO
M_ACT_DIAG = jnp.array([-1., 1., 1., -1., -1., 1., 1., -1., 1., 1., -1., 1.], dtype=jnp.float32)
M_ACT_PERM = jnp.array([4, 5, 6, 7, 0, 1, 2, 3, 9, 8, 10, 11], dtype=jnp.int32)

def mirror_act_jax(a: jnp.ndarray) -> jnp.ndarray:
    return a[..., M_ACT_PERM] * M_ACT_DIAG

def mirror_obs_jax(o: jnp.ndarray) -> jnp.ndarray:
    head = o[..., 0:11] * jnp.array([1., -1., 1., -1., 1., -1., 1., -1., -1., -1., -1.], dtype=jnp.float32)
    pa1  = mirror_act_jax(o[..., 11:23])
    pa2  = mirror_act_jax(o[..., 23:35])
    tof  = o[..., 35:36]
    return jnp.concatenate([head, pa1, pa2, tof], axis=-1)


def quat_to_proj_gravity_jax(q: jnp.ndarray) -> jnp.ndarray:
    w, x, y, z = q[0], q[1], q[2], q[3]
    gx = -2.0 * (x * z - w * y)
    gy = -2.0 * (y * z + w * x)
    gz = -(1.0 - 2.0 * (x * x + y * y))
    return jnp.array([gx, gy, gz], dtype=jnp.float32)


def compute_cpg_prior_jax(phi, cmd_vx, cmd_vy, cmd_yaw, cpg, head_phase):
    speed_mag = jnp.clip(jnp.hypot(cmd_vx / 0.12, cmd_yaw / 0.55), 0.0, 1.35)
    act = jnp.clip(speed_mag, 0.0, 1.0)
    fwd_scale = jnp.clip(cmd_vx / 0.12, -0.85, 1.25)
    yaw_scale = jnp.clip(cmd_yaw / 0.55, -1.0, 1.0)
    lat_scale = jnp.clip(cmd_vy / 0.08, -1.0, 1.0)

    lean_fwd    = cpg[1] * fwd_scale
    level_ratio = cpg[2]
    roll_amp    = cpg[3] * act
    ank_ratio   = cpg[4]
    stride_amp  = cpg[5] * jnp.where(jnp.abs(fwd_scale) > 0.05, fwd_scale, 0.45 * jnp.abs(yaw_scale))
    lift_roll   = cpg[6] * act
    phase_off   = cpg[7]
    arm_amp     = cpg[8] * act
    ank_push    = cpg[9] * act
    turn_diff   = cpg[10] * yaw_scale
    turn_roll   = cpg[11] * yaw_scale

    w_roll = jnp.sin(2.0 * jnp.pi * phi)
    w_step = jnp.cos(2.0 * jnp.pi * (phi + phase_off))
    l_sw = jnp.maximum(0.0, w_roll)
    r_sw = jnp.maximum(0.0, -w_roll)

    # Inner-leg-only shortening (verified bit-exact with simulation/rnv1_policy_core.py & ESP32-S3 C++)
    l_stride_gain = 1.0 - jnp.maximum(0.0, -turn_diff)
    r_stride_gain = 1.0 - jnp.maximum(0.0,  turn_diff)
    rc = roll_amp * w_roll + 0.05 * lat_scale * w_step + turn_roll * w_step

    l_hip = -lean_fwd + stride_amp * l_stride_gain * w_step
    r_hip = -lean_fwd - stride_amp * r_stride_gain * w_step

    q = jnp.array([
        rc + lift_roll * l_sw, l_hip, -l_hip * level_ratio - ank_push * w_roll, ank_ratio * rc - lift_roll * l_sw,
        rc - lift_roll * r_sw, r_hip, -r_hip * level_ratio + ank_push * w_roll, ank_ratio * rc + lift_roll * r_sw,
        -arm_amp * w_step - 0.25 * yaw_scale,
         arm_amp * w_step + 0.25 * yaw_scale,
        jnp.clip(0.45 * yaw_scale + 0.35 * jnp.sin(head_phase), -1.1, 1.1),
        jnp.clip(0.08 + 0.08 * jnp.sin(0.5 * head_phase), -0.35, 0.45),
    ], dtype=jnp.float32)
    return jnp.clip(q, Q_MIN, Q_MAX)


def _glorot_w(key, out_dim: int, in_dim: int, scale: float = 1.0) -> jnp.ndarray:
    std = scale * jnp.sqrt(2.0 / float(in_dim + out_dim))
    return (jax.random.normal(key, (out_dim, in_dim), dtype=jnp.float32) * std)


def init_rnv1_params(rng) -> dict:
    # 100% Pure-JAX parameter dict for RNV1 Asymmetric Actor-Critic (ZERO Flax dependency)
    keys = jax.random.split(rng, 6)
    return {
        "actor": {
            "w1": _glorot_w(keys[0], H1_DIM, OBS_DIM, 1.0),
            "b1": jnp.zeros((H1_DIM,), dtype=jnp.float32),
            "w2": _glorot_w(keys[1], H2_DIM, H1_DIM, 1.0),
            "b2": jnp.zeros((H2_DIM,), dtype=jnp.float32),
            "w3": _glorot_w(keys[2], ACT_DIM, H2_DIM, 0.05),
            "b3": jnp.zeros((ACT_DIM,), dtype=jnp.float32),
            "log_std": jnp.full((ACT_DIM,), -1.4, dtype=jnp.float32),
            "cpg_delta": jnp.zeros((12,), dtype=jnp.float32),
        },
        "critic": {
            "w1": _glorot_w(keys[3], 128, PRIV_DIM, 1.0),
            "b1": jnp.zeros((128,), dtype=jnp.float32),
            "w2": _glorot_w(keys[4], 128, 128, 1.0),
            "b2": jnp.zeros((128,), dtype=jnp.float32),
            "w3": _glorot_w(keys[5], 1, 128, 0.5),
            "b3": jnp.zeros((1,), dtype=jnp.float32),
        },
    }


def apply_rnv1(params: dict, obs: jnp.ndarray, priv_obs: jnp.ndarray):
    # Deployable ESP32-S3 Actor: EXACT 36 -> 64 -> 64 -> 12 (tanh)
    a = params["actor"]
    h1 = jnp.tanh(obs @ a["w1"].T + a["b1"])
    h2 = jnp.tanh(h1  @ a["w2"].T + a["b2"])
    mean_delta = jnp.tanh(h2 @ a["w3"].T + a["b3"])
    cpg = DEFAULT_CPG + jnp.clip(a["cpg_delta"], -0.20, 0.20)

    # Asymmetric Privileged Critic (Training-only, NOT deployed to ESP32-S3)
    c = params["critic"]
    v1 = jax.nn.elu(priv_obs @ c["w1"].T + c["b1"])
    v2 = jax.nn.elu(v1       @ c["w2"].T + c["b2"])
    value = (v2 @ c["w3"].T + c["b3"]).squeeze(-1)
    return mean_delta, a["log_std"], cpg, value


class EnvState(NamedTuple):
    mjx_data: mjx.Data
    obs: jnp.ndarray
    priv_obs: jnp.ndarray
    cmd: jnp.ndarray           # [vx, vy, yaw]
    phi: jnp.ndarray
    head_phase: jnp.ndarray
    prev_act: jnp.ndarray
    prev2_act: jnp.ndarray
    filtered_ctrl: jnp.ndarray
    servo_alpha: jnp.ndarray   # Domain-randomized 1st-order MG90S lag per env
    step_count: jnp.ndarray
    rng: jnp.ndarray


def _make_obs(d: mjx.Data, cmd, phi, prev_act, prev2_act, rng):
    rng, k1, k2 = jax.random.split(rng, 3)
    pg = quat_to_proj_gravity_jax(d.qpos[3:7])
    pg_noisy = pg + jax.random.normal(k1, (3,)) * 0.015
    pg_noisy = pg_noisy / jnp.maximum(jnp.linalg.norm(pg_noisy), 1e-6)
    gyro = d.qvel[3:6] + jax.random.normal(k2, (3,)) * 0.05
    tof_norm = jnp.array([1.0], dtype=jnp.float32)

    obs = jnp.concatenate([
        pg_noisy, gyro * 0.25, cmd,
        jnp.array([jnp.sin(2.0 * jnp.pi * phi), jnp.cos(2.0 * jnp.pi * phi)]),
        prev_act, prev2_act, tof_norm
    ])
    priv = jnp.concatenate([obs, d.qvel[0:3], d.qpos[2:3], d.qpos[7:19]])
    return obs, priv, rng


# Precompute neutral MJX start state once outside JIT for 2x faster resets
_base_d = mjx.make_data(mjx_model)
_base_d = mjx.forward(mjx_model, _base_d.replace(qpos=_base_d.qpos.at[2].set(0.123)))


def reset_env(rng):
    rng, k_vx, k_vy, k_yaw, k_tau, k_q, k_st = jax.random.split(rng, 7)
    qpos = _base_d.qpos.at[7:].set(jax.random.uniform(k_q, (ACT_DIM,), minval=-0.025, maxval=0.025))
    d = _base_d.replace(qpos=qpos, qvel=jnp.zeros_like(_base_d.qvel), ctrl=jnp.zeros_like(_base_d.ctrl))

    is_stand = jax.random.uniform(k_st, ()) < 0.15
    raw_cmd = jnp.array([
        jax.random.uniform(k_vx,  (), minval=-0.02, maxval=0.18),
        jax.random.uniform(k_vy,  (), minval=-0.04, maxval=0.04),
        jax.random.uniform(k_yaw, (), minval=-0.65, maxval=0.65),
    ], dtype=jnp.float32)
    cmd = jnp.where(is_stand, jnp.zeros(3, dtype=jnp.float32), raw_cmd)
    tau_s = jax.random.uniform(k_tau, (), minval=0.018, maxval=0.036)
    alpha = 0.002 / (tau_s + 0.002)

    prev_act = jnp.zeros(ACT_DIM, dtype=jnp.float32)
    obs, priv, rng = _make_obs(d, cmd, jnp.float32(0.0), prev_act, prev_act, rng)
    return EnvState(
        mjx_data=d, obs=obs, priv_obs=priv, cmd=cmd,
        phi=jnp.float32(0.0), head_phase=jnp.float32(0.0),
        prev_act=prev_act, prev2_act=prev_act, filtered_ctrl=prev_act,
        servo_alpha=alpha, step_count=jnp.int32(0), rng=rng
    )


def step_env(state: EnvState, action_delta: jnp.ndarray, cpg: jnp.ndarray, curriculum_scale: float):
    prior = compute_cpg_prior_jax(state.phi, state.cmd[0], state.cmd[1], state.cmd[2], cpg, state.head_phase)
    scale_vec = jnp.where(jnp.arange(ACT_DIM) >= 10, RESIDUAL_SCALE * 0.25, RESIDUAL_SCALE)
    target_q = jnp.clip(prior + jnp.clip(action_delta, -1.0, 1.0) * scale_vec, Q_MIN, Q_MAX)

    # Random external push perturbation (curriculum-scaled up to 3.2N for 20ms!)
    rng, k_push, k_f = jax.random.split(state.rng, 3)
    do_push = (jax.random.uniform(k_push, ()) < (0.015 * curriculum_scale))
    push_f = jax.random.uniform(k_f, (3,), minval=-3.2, maxval=3.2) * curriculum_scale * jnp.array([1.0, 1.0, 0.2])
    xfrc = state.mjx_data.xfrc_applied.at[1, 0:3].set(jnp.where(do_push, push_f, 0.0))
    d0 = state.mjx_data.replace(xfrc_applied=xfrc)

    def substep_fn(carry, _):
        d, f_ctrl = carry
        f_ctrl = f_ctrl + state.servo_alpha * (target_q - f_ctrl)
        d = mjx.step(mjx_model, d.replace(ctrl=f_ctrl))
        return (d, f_ctrl), jnp.mean(d.actuator_force ** 2)

    (d_next, f_ctrl_next), trq_sq = jax.lax.scan(substep_fn, (d0, state.filtered_ctrl), None, length=PHYSICS_SUBSTEPS)

    # Compute dense biomechanically grounded reward
    pg = quat_to_proj_gravity_jax(d_next.qpos[3:7])
    tilt = jnp.arccos(jnp.clip(-pg[2], -1.0, 1.0))
    z_torso = d_next.qpos[2]
    fell = (z_torso < 0.068) | (tilt > 0.75)

    r_vx = 3.0 * jnp.exp(-((d_next.qvel[0] - state.cmd[0]) / 0.09) ** 2)
    r_vy = 1.0 * jnp.exp(-((d_next.qvel[1] - state.cmd[1]) / 0.08) ** 2)
    r_wz = 2.0 * jnp.exp(-((d_next.qvel[5] - state.cmd[2]) / 0.35) ** 2)
    r_up = 1.5 * jnp.exp(-(tilt / 0.20) ** 2)
    r_smooth = -0.35 * jnp.mean((target_q - state.prev_act) ** 2)
    r_energy = -0.40 * jnp.mean(trq_sq)
    reward = jnp.where(fell, -15.0, r_vx + r_vy + r_wz + r_up + r_smooth + r_energy)

    next_step = state.step_count + 1
    done = fell | (next_step >= EPISODE_STEPS)

    freq_hz = jnp.clip(cpg[0], 0.9, 2.5)
    next_phi = jnp.mod(state.phi + freq_hz * CONTROL_DT, 1.0)
    next_hp = state.head_phase + 1.4 * CONTROL_DT
    obs, priv, rng = _make_obs(d_next, state.cmd, next_phi, target_q, state.prev_act, rng)

    next_state = EnvState(
        mjx_data=d_next, obs=obs, priv_obs=priv, cmd=state.cmd,
        phi=next_phi, head_phase=next_hp,
        prev_act=target_q, prev2_act=state.prev_act, filtered_ctrl=f_ctrl_next,
        servo_alpha=state.servo_alpha, step_count=next_step, rng=rng
    )

    reset_st = reset_env(rng)
    out_state = jax.tree_util.tree_map(lambda a, b: jnp.where(done, a, b), reset_st, next_state)
    return out_state, reward, done

print("[OK] Pure-JAX RNV1 MJX parallel GPU environment & Asymmetric Actor-Critic ready!")


In [ ]:
#@title 5. START EXTREME RNV1 TRAINING ON A100 (Pure-JAX PPO + GAE + 3-Phase Curriculum) { display-mode: "form" }
ROLLOUT_LEN = 32
PPO_EPOCHS = 4
MINIBATCHES = 16
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.2
ENT_COEF = 0.005
VF_COEF = 0.5
LR_INIT = 4e-4
LR_MIN_RATIO = 0.05
MAX_GRAD_NORM = 0.8

steps_per_iter = NUM_ENVS * ROLLOUT_LEN
num_iters = max(4, TOTAL_TIMESTEPS // steps_per_iter)
total_opt_steps = float(num_iters * PPO_EPOCHS * MINIBATCHES)
print(f"[RNV1] Launching Pure-JAX PPO | Envs={NUM_ENVS} | Steps/Iter={steps_per_iter:,} | Iters={num_iters} | Target={num_iters*steps_per_iter:,} steps")

rng = jax.random.PRNGKey(SEED)
rng, init_k, env_k = jax.random.split(rng, 3)
params = init_rnv1_params(init_k)


def init_adam_state(p):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, p),
        "v": jax.tree_util.tree_map(jnp.zeros_like, p),
        "step": jnp.int32(0),
    }


def adam_step(p, grads, state):
    # Global-norm gradient clipping
    leaves = jax.tree_util.tree_leaves(grads)
    gnorm = jnp.sqrt(sum(jnp.sum(g * g) for g in leaves) + 1e-12)
    clip_coef = jnp.minimum(1.0, MAX_GRAD_NORM / gnorm)
    grads = jax.tree_util.tree_map(lambda g: g * clip_coef, grads)

    step = state["step"] + 1
    progress = jnp.minimum(1.0, step.astype(jnp.float32) / jnp.maximum(total_opt_steps, 1.0))
    lr = LR_INIT * (LR_MIN_RATIO + 0.5 * (1.0 - LR_MIN_RATIO) * (1.0 + jnp.cos(jnp.pi * progress)))

    b1, b2, eps = 0.9, 0.999, 1e-8
    m = jax.tree_util.tree_map(lambda m_, g: b1 * m_ + (1.0 - b1) * g, state["m"], grads)
    v = jax.tree_util.tree_map(lambda v_, g: b2 * v_ + (1.0 - b2) * (g * g), state["v"], grads)
    m_hat_scale = 1.0 / (1.0 - b1 ** step.astype(jnp.float32))
    v_hat_scale = 1.0 / (1.0 - b2 ** step.astype(jnp.float32))
    p_new = jax.tree_util.tree_map(
        lambda p_, m_, v_: p_ - lr * (m_ * m_hat_scale) / (jnp.sqrt(v_ * v_hat_scale) + eps),
        p, m, v
    )
    return p_new, {"m": m, "v": v, "step": step}


opt_state = init_adam_state(params)
v_reset = jax.jit(jax.vmap(reset_env))
v_step = jax.jit(jax.vmap(step_env, in_axes=(0, 0, None, None)))


@jax.jit
def collect_and_update(params, opt_state, env_states, rng, curriculum_scale):
    def rollout_step(carry, _):
        st, key = carry
        key, act_k = jax.random.split(key)
        mean, log_std, cpg, val = apply_rnv1(params, st.obs, st.priv_obs)
        std = jnp.exp(jnp.clip(log_std, -3.0, 0.5))
        act = mean + std * jax.random.normal(act_k, mean.shape)
        logp = -0.5 * jnp.sum(((act - mean) / std) ** 2 + 2.0 * jnp.log(std) + jnp.log(2.0 * jnp.pi), axis=-1)
        next_st, rew, done = v_step(st, act, cpg, curriculum_scale)
        return (next_st, key), (st.obs, st.priv_obs, act, logp, rew, done, val)

    (env_states, rng), traj = jax.lax.scan(rollout_step, (env_states, rng), None, length=ROLLOUT_LEN)
    obs_b, priv_b, act_b, logp_b, rew_b, done_b, val_b = traj
    _, _, _, last_val = apply_rnv1(params, env_states.obs, env_states.priv_obs)

    # GAE Advantage Computation
    def gae_step(carry, xs):
        gae, next_v = carry
        r, d, v = xs
        delta = r + GAMMA * next_v * (1.0 - d) - v
        gae = delta + GAMMA * GAE_LAMBDA * (1.0 - d) * gae
        return (gae, v), gae

    _, adv_b = jax.lax.scan(gae_step, (jnp.zeros(NUM_ENVS), last_val), (rew_b, done_b.astype(jnp.float32), val_b), reverse=True)
    ret_b = adv_b + val_b

    B = ROLLOUT_LEN * NUM_ENVS
    obs_f  = obs_b.reshape((B, OBS_DIM))
    priv_f = priv_b.reshape((B, PRIV_DIM))
    act_f  = act_b.reshape((B, ACT_DIM))
    logp_f = logp_b.reshape((B,))
    adv_f  = adv_b.reshape((B,))
    adv_f  = (adv_f - jnp.mean(adv_f)) / (jnp.std(adv_f) + 1e-8)
    ret_f  = ret_b.reshape((B,))

    obs_stand_ref = jnp.zeros((1, OBS_DIM), dtype=jnp.float32).at[0, 2].set(-1.0).at[0, 10].set(1.0).at[0, 35].set(1.0)
    priv_stand_ref = jnp.zeros((1, PRIV_DIM), dtype=jnp.float32)

    def loss_fn(p, o, pr, a, lp_old, adv, ret):
        mean, log_std, cpg_cur, val = apply_rnv1(p, o, pr)
        std = jnp.exp(jnp.clip(log_std, -3.0, 0.5))
        lp = -0.5 * jnp.sum(((a - mean) / std) ** 2 + 2.0 * jnp.log(std) + jnp.log(2.0 * jnp.pi), axis=-1)
        ratio = jnp.exp(lp - lp_old)
        pg1 = -adv * ratio
        pg2 = -adv * jnp.clip(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)
        pg_loss = jnp.mean(jnp.maximum(pg1, pg2))
        v_loss = 0.5 * jnp.mean((val - ret) ** 2)
        ent = jnp.mean(jnp.sum(log_std + 0.5 * jnp.log(2.0 * jnp.pi * jnp.e), axis=-1))
        # Bilateral L/R mirror symmetry + static stand zero-drift + CPG L2 anchor
        mean_mir, _, _, _ = apply_rnv1(p, mirror_obs_jax(o), pr)
        sym_loss = jnp.mean((mirror_act_jax(mean) - mean_mir) ** 2)
        mean_st, _, _, _ = apply_rnv1(p, obs_stand_ref, priv_stand_ref)
        stand_loss = jnp.mean(mean_st ** 2) + 0.01 * jnp.mean((cpg_cur - DEFAULT_CPG) ** 2)
        return pg_loss + VF_COEF * v_loss - ENT_COEF * ent + 0.08 * sym_loss + 0.05 * stand_loss

    grad_fn = jax.value_and_grad(loss_fn)

    def epoch_step(carry, _):
        p, os_, key = carry
        key, perm_k = jax.random.split(key)
        perm = jax.random.permutation(perm_k, B).reshape((MINIBATCHES, -1))
        def mb_step(c2, idx):
            p2, os2 = c2
            loss, grads = grad_fn(p2, obs_f[idx], priv_f[idx], act_f[idx], logp_f[idx], adv_f[idx], ret_f[idx])
            p2, os2 = adam_step(p2, grads, os2)
            return (p2, os2), loss
        (p, os_), losses = jax.lax.scan(mb_step, (p, os_), perm)
        return (p, os_, key), jnp.mean(losses)

    (params, opt_state, rng), mean_loss = jax.lax.scan(epoch_step, (params, opt_state, rng), None, length=PPO_EPOCHS)
    return params, opt_state, env_states, rng, jnp.mean(rew_b), jnp.mean(done_b.astype(jnp.float32)), jnp.mean(mean_loss)


env_states = v_reset(jax.random.split(env_k, NUM_ENVS))
print("[RNV1] Compiling XLA GPU graph (1st iteration takes ~25-45s, then runs at peak A100 speed)...")
t_start = time.time()
hist_steps, hist_rew, hist_fall = [], [], []
best_rew = -1e9
best_params = params

for it in range(num_iters):
    curr_scale = float(np.clip((it + 1) / max(1, int(0.6 * num_iters)), 0.15, 1.0))
    params, opt_state, env_states, rng, m_rew, m_fall, m_loss = collect_and_update(
        params, opt_state, env_states, rng, curr_scale
    )
    if it == 0:
        t_after_jit = time.time()
    rew_val = float(m_rew)
    if rew_val > best_rew:
        best_rew = rew_val
        best_params = params
    if (it + 1) % max(1, num_iters // 20) == 0 or it == 0 or it == num_iters - 1:
        steps_done = (it + 1) * steps_per_iter
        elapsed = max(time.time() - t_start, 1e-3)
        sps = steps_done / elapsed
        eta_min = ((num_iters - it - 1) * steps_per_iter / max(sps, 1.0)) / 60.0
        hist_steps.append(steps_done / 1e6)
        hist_rew.append(rew_val)
        hist_fall.append(float(m_fall) * 100.0)
        print(f"Iter {it+1:4d}/{num_iters} | Steps: {steps_done/1e6:6.2f}M | {sps:9,.0f} SPS | "
              f"Reward: {rew_val:+6.3f} | Fall%: {float(m_fall)*100:4.2f}% | Curr: {curr_scale:.2f} | ETA: {eta_min:4.1f}m")

params = best_params
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1); plt.plot(hist_steps, hist_rew, lw=2); plt.title("RNV1 Mean Step Reward"); plt.xlabel("Million Steps"); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2); plt.plot(hist_steps, hist_fall, color="crimson", lw=2); plt.title("RNV1 Fall Rate (%)"); plt.xlabel("Million Steps"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
#@title 6. High-Precision MuJoCo Verification & Simulation Video Playback { display-mode: "form" }
a_dict = params["actor"]
W1 = np.array(a_dict["w1"], dtype=np.float32)  # (64, 36)
B1 = np.array(a_dict["b1"], dtype=np.float32)
W2 = np.array(a_dict["w2"], dtype=np.float32)  # (64, 64)
B2 = np.array(a_dict["b2"], dtype=np.float32)
W3 = np.array(a_dict["w3"], dtype=np.float32)  # (12, 64)
B3 = np.array(a_dict["b3"], dtype=np.float32)
CPG_OPT = np.array(DEFAULT_CPG + np.clip(a_dict["cpg_delta"], -0.20, 0.20), dtype=np.float32)

def infer_rnv1_np(obs, phi, cmd_vx, cmd_vy, cmd_yaw, hp):
    prior = np.array(compute_cpg_prior_jax(phi, cmd_vx, cmd_vy, cmd_yaw, CPG_OPT, hp), dtype=np.float32)
    h1 = np.tanh(W1 @ obs + B1)
    h2 = np.tanh(W2 @ h1 + B2)
    delta = np.tanh(W3 @ h2 + B3) * RESIDUAL_SCALE
    delta[10:] *= 0.25
    return np.clip(prior + delta, np.array(Q_MIN), np.array(Q_MAX))

mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetData(mj_model, mj_data)
mj_data.qpos[2] = 0.123
mujoco.mj_forward(mj_model, mj_data)

renderer = None
try:
    renderer = mujoco.Renderer(mj_model, height=360, width=640)
except Exception as e:
    print(f"[NOTE] Headless OpenGL renderer unavailable ({e}); continuing physics evaluation without MP4 video.")

frames = []
phi, hp = 0.0, 0.0
prev_act = np.zeros(ACT_DIM, dtype=np.float32)
prev2_act = np.zeros(ACT_DIM, dtype=np.float32)
f_ctrl = np.zeros(ACT_DIM, dtype=np.float32)
alpha = 0.002 / (0.025 + 0.002)

for step in range(250):  # 5.0 seconds @ 50Hz
    q = mj_data.qpos[3:7]
    pg = np.array(quat_to_proj_gravity_jax(q), dtype=np.float32)
    gyro = mj_data.qvel[3:6].astype(np.float32)
    obs = np.concatenate([pg, gyro * 0.25, [0.12, 0.0, 0.0], [np.sin(2*np.pi*phi), np.cos(2*np.pi*phi)], prev_act, prev2_act, [1.0]]).astype(np.float32)
    target_q = infer_rnv1_np(obs, phi, 0.12, 0.0, 0.0, hp)
    prev2_act, prev_act = prev_act.copy(), target_q.copy()
    for _ in range(10):
        f_ctrl = f_ctrl + alpha * (target_q - f_ctrl)
        mj_data.ctrl[:] = f_ctrl
        mujoco.mj_step(mj_model, mj_data)
    phi = (phi + float(np.clip(CPG_OPT[0], 0.9, 2.5)) * 0.02) % 1.0
    hp += 1.4 * 0.02
    if renderer is not None and step % 2 == 0:
        renderer.update_scene(mj_data)
        frames.append(renderer.render())

print(f"[EVAL] 5.0s Walk Distance: {mj_data.qpos[0]:.3f} m | Torso Z: {mj_data.qpos[2]:.3f} m | Peak Torque: {np.max(np.abs(mj_data.actuator_force)):.4f} N-m")
if frames:
    try:
        import mediapy as media
        media.write_video("RNV1_walk_eval.mp4", frames, fps=25)
        media.show_video(frames, fps=25)
    except Exception as e:
        print(f"[NOTE] Video display skipped: {e}")


In [ ]:
#@title 7. Auto-Export ESP32-S3 Firmware Header (rnv1_policy_weights.h) & Download { display-mode: "form" }
import json, zipfile

def fmt_c_arr(name, arr):
    flat = np.asarray(arr, dtype=np.float32).ravel()
    vals = ", ".join(f"{float(v):.7e}f" for v in flat)
    return f"alignas(16) static const float {name}[{len(flat)}] = {{\n  {vals}\n}};\n"

header_lines = [
    f"// AUTO-GENERATED {TRAINING_VERSION} COLAB PRO A100 EXTREME RL WEIGHTS",
    "// Drop directly into firmware/include/rnv1_policy_weights.h and flash XIAO ESP32S3 Sense!",
    "#pragma once",
    "#include <stdint.h>",
    f"#define RNV1_OBS_DIM {OBS_DIM}",
    f"#define RNV1_H1_DIM  {H1_DIM}",
    f"#define RNV1_H2_DIM  {H2_DIM}",
    f"#define RNV1_ACT_DIM {ACT_DIM}",
    f"#define RNV1_RESIDUAL_SCALE {RESIDUAL_SCALE:.7e}f",
    fmt_c_arr("RNV1_Q_MIN", np.array(Q_MIN)),
    fmt_c_arr("RNV1_Q_MAX", np.array(Q_MAX)),
    fmt_c_arr("RNV1_CPG", CPG_OPT),
    fmt_c_arr("RNV1_W1", W1),
    fmt_c_arr("RNV1_B1", B1),
    fmt_c_arr("RNV1_W2", W2),
    fmt_c_arr("RNV1_B2", B2),
    fmt_c_arr("RNV1_W3", W3),
    fmt_c_arr("RNV1_B3", B3),
]
Path("rnv1_policy_weights.h").write_text("\n".join(header_lines), encoding="utf-8")

weights_json = {
    "version": TRAINING_VERSION, "obs_dim": OBS_DIM, "h1_dim": H1_DIM, "h2_dim": H2_DIM,
    "act_dim": ACT_DIM, "control_hz": CONTROL_HZ, "residual_scale": RESIDUAL_SCALE,
    "cpg": CPG_OPT.tolist(), "w1": W1.tolist(), "b1": B1.tolist(),
    "w2": W2.tolist(), "b2": B2.tolist(), "w3": W3.tolist(), "b3": B3.tolist(),
}
Path("rnv1_policy_weights.json").write_text(json.dumps(weights_json, indent=2), encoding="utf-8")

bundle = "RNV1_Trained_Bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in ["rnv1_policy_weights.h", "rnv1_policy_weights.json", "RNV1_walk_eval.mp4"]:
        if Path(fn).exists():
            zf.write(fn)

print(f"[EXPORT COMPLETE] {bundle} ready ({Path(bundle).stat().st_size / 1024:.1f} KB)!")
try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    print("Running outside Colab: files saved locally in current directory.")
